# Geospatial Data Science with Uber's H3
### Hands-on lab · 20 minutes

**The question:** *what is a block of houses worth — given what's around it?*

That is the shape of most production geospatial problems: given where a thing is, and what
surrounds it, predict what happens next. The model is the easy part. The expensive part is
computing **"what's around it"** twenty thousand times.

We will do exactly that job twice — first the traditional way, then with H3 — on the same data,
answering the same two questions:

> **Q1 — proximity.** For every block, which other blocks lie within 5 km, and what are they worth?
> **Q2 — density.** Aggregate everything into areas and map where value concentrates.

Then we'll feed the answer to a model and see whether it was worth the trouble.

*Run every cell in order. Nothing here needs an API key, a login, or an install beyond one line.*

---
## Part 0 · Setup

In [ ]:
# h3 is the only thing Colab doesn't already have
!pip install -q h3==4.5.0

In [ ]:
import math, time
from collections import defaultdict

import numpy as np
import pandas as pd
import h3
import folium
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error

print("h3 version:", h3.__version__)

### The data — California housing

20,640 census block groups. Each row has a latitude, a longitude, six ordinary numeric columns,
and the thing we want to predict: `median_house_value`.

It loads in one line from a public URL. No auth, no download button.

In [ ]:
LAB = ("https://raw.githubusercontent.com/litandlatte/tanacloud.com/"
       "geospatial-data-science-with-ubers-h3/labs/geospatial-data-science-with-ubers-h3")

URLS = [
    "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv",
    f"{LAB}/data/housing.csv",     # our own mirror, in case the one above ever moves
]

df = None
for u in URLS:
    try:
        df = pd.read_csv(u)
        print("loaded from:", u)
        break
    except Exception as e:
        print("failed:", u, "->", type(e).__name__)
if df is None:
    raise SystemExit("no dataset URL reachable")

df = df.dropna(subset=["median_house_value", "latitude", "longitude"]).reset_index(drop=True)
n = len(df)
print(f"\n{n:,} rows x {df.shape[1]} columns")
df.head()

In [ ]:
# pull the three arrays we'll use constantly
lat = df["latitude"].to_numpy(float)
lon = df["longitude"].to_numpy(float)
val = df["median_house_value"].to_numpy(float)

plt.figure(figsize=(7, 8))
plt.scatter(lon, lat, c=val, s=2, cmap="viridis")
plt.colorbar(label="median house value ($)")
plt.title(f"{n:,} block groups — the only location signal we have is a pair of floats")
plt.xlabel("longitude"); plt.ylabel("latitude")
plt.show()

California is instantly recognisable to us — but **not to a model**. To a model, `latitude` and
`longitude` are just two more numeric columns. It has no idea that two rows 200 m apart are
neighbours, and two rows with similar numbers might be 500 km apart.

---
## Part 1 · The baseline — a model with no sense of place

Six non-spatial columns, a gradient-boosted tree, a 75/25 split. **This is the number to beat.**

In [ ]:
BASE = ["median_income", "housing_median_age", "total_rooms",
        "total_bedrooms", "population", "households"]

tr, te = train_test_split(np.arange(n), test_size=0.25, random_state=42)
TRAIN = set(tr.tolist())

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

model = HistGradientBoostingRegressor(random_state=0).fit(df.loc[tr, BASE], val[tr])
BASE_RMSE = rmse(val[te], model.predict(df.loc[te, BASE]))

print(f"BASELINE RMSE = ${BASE_RMSE:,.0f}")
print("(no spatial features at all)")

Off by about **$66,000** per block. The single biggest thing this model is missing is the one
thing an estate agent would lead with: **what the neighbourhood is like.**

So let's build that feature. For each block: *how many other blocks are within 5 km, and what is
their average value?*

---
## Part 2 · Round 1 — the traditional way

No special tools. Just coordinates and the distance formula.

In [ ]:
R_EARTH = 6371.0  # km

def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in km. Works on scalars or numpy arrays."""
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = p2 - p1
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlam / 2) ** 2
    return 2 * R_EARTH * np.arcsin(np.sqrt(a))

# sanity check on a distance we can look up: Chennai -> Bengaluru is ~290 km
print(f"Chennai -> Bengaluru: {haversine(13.0827, 80.2707, 12.9716, 77.5946):.1f} km")

### The cost of asking "what's within 5 km?"

To answer it for **every** block, every block must be compared with every other block.

In [ ]:
print(f"rows              : {n:,}")
print(f"pairs to check    : {n*n:,}")
print(f"as a float matrix : {n*n*8/1e9:.2f} GB   <- this alone would end the session")

**426 million distance calculations.** Let's find out what that actually costs by writing the
loop the way you'd write it the first time — plain Python, one pair at a time — and timing it on
just **500 rows**.

In [ ]:
def haversine_py(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = p2 - p1
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlam / 2) ** 2
    return 2 * R_EARTH * math.asin(math.sqrt(a))

RADIUS_KM = 5.0
SAMPLE = 500

t0 = time.perf_counter()
for i in range(SAMPLE):
    c = 0
    for j in range(n):
        if i != j and haversine_py(lat[i], lon[i], lat[j], lon[j]) <= RADIUS_KM:
            c += 1
elapsed = time.perf_counter() - t0

per_pair = elapsed / (SAMPLE * n)
full = per_pair * n * n
print(f"{SAMPLE} rows against all {n:,} = {SAMPLE*n:,} pairs in {elapsed:.1f}s")
print(f"  -> {per_pair*1e6:.2f} microseconds per pair")
print(f"  -> all {n*n:,} pairs would take {full:.0f}s = {full/60:.1f} MINUTES")

Minutes — for one feature, on twenty thousand rows, which is a *small* dataset.

We can rescue this with numpy: instead of one pair at a time, do one **row against all rows** as
a single vector operation. Same 426 million calculations, far less Python overhead.

In [ ]:
t0 = time.perf_counter()
nb_count_bf = np.zeros(n)
nb_mean_bf = np.full(n, np.nan)

for i in range(n):
    d = haversine(lat[i], lon[i], lat, lon)
    m = d <= RADIUS_KM
    m[i] = False                      # a block is not its own neighbour
    k = int(m.sum())
    nb_count_bf[i] = k
    if k:
        nb_mean_bf[i] = val[m].mean()

BRUTE_SECONDS = time.perf_counter() - t0
print(f"brute force, all {n:,} rows: {BRUTE_SECONDS:.1f}s")
print(f"median neighbours within {RADIUS_KM:.0f} km: {np.median(nb_count_bf):.0f}")

Seconds instead of minutes — but notice what we did **not** fix. It is still 426 million distance
calculations. We made each one cheaper; we did not make any of them unnecessary. Double the data
and this gets **four times** slower. Hold that thought.

### Q2 the traditional way — round the coordinates

For the map we need areas, not points. The classic trick with no tools: **round the coordinates**
and call everything that lands on the same rounded pair one "cell".

In [ ]:
grid_lat = np.round(lat, 2)          # 0.01 degrees ~ 1 km
grid_lon = np.round(lon, 2)
grid_key = pd.Series([f"{a},{o}" for a, o in zip(grid_lat, grid_lon)])

print(f"{grid_key.nunique():,} distinct grid cells")
print("\nIt works. But look at what kind of cell we just invented:")

In [ ]:
# how big is a 0.01-degree box, and does it stay that size?
print(f"{'latitude':>10}{'width km':>10}{'height km':>11}{'aspect':>9}")
for L in (0, 32.5, 37.0, 42.0, 60, 80):
    w = 111.32 * math.cos(math.radians(L)) * 0.01
    h = 110.574 * 0.01
    print(f"{L:>10}{w:>10.3f}{h:>11.3f}{w/h:>9.3f}")

print("\nAcross California alone the cell narrows from 0.94 km to 0.83 km.")
print("Take it global and it collapses: at 80 deg north it is a sliver.")

That is the famous problem — and honestly, **within a single state it is fairly mild**. Do not
oversell it. The problem that actually bites is the next one.

### The real defect: what does "next to" mean?

In [ ]:
# in a square grid, the 8 surrounding cells are NOT all the same distance away
L = 37.0
w = 111.32 * math.cos(math.radians(L)) * 0.01
h = 110.574 * 0.01
square_nb = sorted([w, w, h, h] + [math.hypot(w, h)] * 4)

print("Square grid — distances to the 8 surrounding cells (km):")
print("  ", [round(x, 3) for x in square_nb])
print(f"   nearest {min(square_nb):.3f}  farthest {max(square_nb):.3f}"
      f"  -> spread {max(square_nb)/min(square_nb):.3f}x")
print()
print("So 'one step away' means four different distances depending on direction,")
print("and you must decide whether the diagonals even count. Every neighbourhood")
print("statistic you build on this grid inherits that ambiguity.")

---
## Part 3 · Round 2 — the same two questions, with H3

H3 assigns every point on Earth to a hexagonal cell, at 16 resolutions. The cell is a single
value — so a location stops being two floats you can only measure, and becomes a **key you can
group by and join on.**

In [ ]:
RES = 7

t0 = time.perf_counter()
cells = np.array([h3.latlng_to_cell(a, o, RES) for a, o in zip(lat, lon)])
index_seconds = time.perf_counter() - t0

print(f"indexed all {n:,} rows in {index_seconds:.3f}s")
print(f"{len(set(cells)):,} distinct cells at resolution {RES}")
print("\nfirst five rows:")
for i in range(5):
    print(f"  ({lat[i]:.4f}, {lon[i]:.4f})  ->  {cells[i]}")

In [ ]:
# what one cell actually is
c = cells[0]
print(f"cell            : {c}")
print(f"as an integer   : {int(c, 16)}   <- 8 bytes, same as ONE of the two floats it replaces")
print(f"resolution      : {h3.get_resolution(c)}")
print(f"area            : {h3.cell_area(c, unit='km^2'):.4f} km2")
print(f"average edge    : {h3.average_hexagon_edge_length(RES, unit='km')*1000:.0f} m")
print(f"centre          : {tuple(round(x, 5) for x in h3.cell_to_latlng(c))}")

### Q2 — density, in one groupby

This is the whole of question 2. A `groupby` on a column that didn't exist ninety seconds ago.

In [ ]:
t0 = time.perf_counter()
agg = (pd.DataFrame({"cell": cells, "value": val})
         .groupby("cell")["value"]
         .agg(["mean", "count"]))
print(f"aggregated in {time.perf_counter()-t0:.3f}s -> {len(agg):,} cells")
agg.sort_values("count", ascending=False).head()

In [ ]:
# keep cells with at least 3 points -- a mean over one point is not a mean
shown = agg[agg["count"] >= 3]
print(f"mapping {len(shown):,} cells (of {len(agg):,})")

features = [{
    "type": "Feature",
    "id": cell,
    "geometry": {"type": "Polygon",
                 "coordinates": [[[lng, la] for la, lng in h3.cell_to_boundary(cell)]]},
    "properties": {"cell": cell, "mean": float(row["mean"])},
} for cell, row in shown.iterrows()]

m = folium.Map(location=[37.0, -119.5], zoom_start=6, tiles="cartodbpositron")
folium.Choropleth(
    geo_data={"type": "FeatureCollection", "features": features},
    data=shown.reset_index(),
    columns=["cell", "mean"],
    key_on="feature.id",
    fill_color="YlOrRd",
    fill_opacity=0.75,
    line_opacity=0.2,
    nan_fill_opacity=0,
    legend_name=f"mean house value ($) — H3 resolution {RES}",
).add_to(m)
m

### Q1 — proximity, and what "one step away" means now

In [ ]:
c = h3.latlng_to_cell(37.0, -120.0, RES)
c_lat, c_lon = h3.cell_to_latlng(c)

d = sorted(float(haversine(c_lat, c_lon, *h3.cell_to_latlng(nb)))
           for nb in h3.grid_disk(c, 1) if nb != c)

print(f"H3 res {RES} — distances to the {len(d)} surrounding cells (km):")
print("  ", [round(x, 3) for x in d])
print(f"   nearest {min(d):.3f}  farthest {max(d):.3f}  -> spread {max(d)/min(d):.3f}x")
print()
print(f"   square grid was {max(square_nb)/min(square_nb):.3f}x across 8 ambiguous neighbours.")
print("   Six neighbours, one shared edge each, all the same distance. No diagonals to argue about.")

**That** is why hexagons — not equal area, which is oversold. It is that a hexagon has exactly
six neighbours and every one of them is the same step away, so `grid_disk(cell, k)` is a genuine
distance ring.

### The hybrid: let H3 do the filtering, let haversine do the measuring

We still want the *exact* 5 km answer. H3 doesn't replace the distance formula — it removes the
need to run it 426 million times. Only cells that could possibly be in range are considered;
haversine then measures the handful of survivors.

In [ ]:
edge_km = h3.average_hexagon_edge_length(RES, unit="km")
K = math.ceil(RADIUS_KM / (math.sqrt(3) * edge_km)) + 1
print(f"radius {RADIUS_KM:.0f} km at res {RES} (edge {edge_km:.3f} km) -> need k={K} rings\n")

members = defaultdict(list)
for i, cell in enumerate(cells):
    members[cell].append(i)

t0 = time.perf_counter()
nb_count_h3 = np.zeros(n)
nb_mean_h3 = np.full(n, np.nan)
disk_cache = {}

for i in range(n):
    cell = cells[i]
    cand = disk_cache.get(cell)
    if cand is None:                                  # cells in the same hex share a disk
        cand = np.asarray([j for nb in h3.grid_disk(cell, K) for j in members.get(nb, ())])
        disk_cache[cell] = cand
    d = haversine(lat[i], lon[i], lat[cand], lon[cand])
    m_ = (d <= RADIUS_KM) & (cand != i)
    k = int(m_.sum())
    nb_count_h3[i] = k
    if k:
        nb_mean_h3[i] = val[cand[m_]].mean()

HYBRID_SECONDS = time.perf_counter() - t0
print(f"brute force : {BRUTE_SECONDS:6.2f}s")
print(f"H3 hybrid   : {HYBRID_SECONDS:6.2f}s   ->  {BRUTE_SECONDS/HYBRID_SECONDS:.1f}x faster")

In [ ]:
# the only thing that matters: is it the SAME answer?
same_count = bool((nb_count_h3 == nb_count_bf).all())
same_mean = bool(np.allclose(np.nan_to_num(nb_mean_h3, nan=-1),
                             np.nan_to_num(nb_mean_bf, nan=-1)))

print(f"neighbour counts identical : {'YES' if same_count else 'NO'}")
print(f"neighbour means  identical : {'YES' if same_mean else 'NO'}")
print("\nNot an approximation. The same answer, with most of the work deleted.")

### And the gap widens with scale

The speed-up isn't a fixed discount. Brute force is O(n²); the H3 route is roughly linear.

In [ ]:
def brute_upto(N):
    t0 = time.perf_counter()
    for i in range(N):
        d = haversine(lat[i], lon[i], lat[:N], lon[:N])
        mm = d <= RADIUS_KM; mm[i] = False
        int(mm.sum())
    return time.perf_counter() - t0

def hybrid_upto(N):
    t0 = time.perf_counter()
    cl = np.array([h3.latlng_to_cell(a, o, RES) for a, o in zip(lat[:N], lon[:N])])
    mem = defaultdict(list)
    for i, cc in enumerate(cl):
        mem[cc].append(i)
    cache = {}
    for i in range(N):
        cc = cl[i]
        cand = cache.get(cc)
        if cand is None:
            cand = np.asarray([j for nb in h3.grid_disk(cc, K) for j in mem.get(nb, ())])
            cache[cc] = cand
        d = haversine(lat[i], lon[i], lat[cand], lon[cand])
        mm = (d <= RADIUS_KM) & (cand != i)
        int(mm.sum())
    return time.perf_counter() - t0

sizes = [2500, 5000, 10000, n]
rows = []
print(f"{'rows':>8}{'brute s':>10}{'H3 s':>9}{'speed-up':>10}")
for N in sizes:
    b, hy = brute_upto(N), hybrid_upto(N)
    rows.append((N, b, hy))
    print(f"{N:>8}{b:>10.2f}{hy:>9.2f}{b/hy:>9.1f}x")

plt.figure(figsize=(7, 4))
plt.plot([r[0] for r in rows], [r[1] for r in rows], "o-", label="brute force  O(n^2)")
plt.plot([r[0] for r in rows], [r[2] for r in rows], "o-", label="H3 hybrid  ~O(n)")
plt.xlabel("rows"); plt.ylabel("seconds"); plt.legend()
plt.title("The gap is not a discount — it grows")
plt.grid(alpha=.3); plt.show()

---
## Part 4 · Round 3 — was any of this worth it?

Speed is only interesting if the feature earns its place. Turn "what's around it" into two
columns and re-run the model.

**One rule that matters more than any of the code above:** the neighbourhood statistic is built
from **training rows only**, and a row is **excluded from its own neighbourhood**. Skip either and
the row gets to see its own answer, the score looks wonderful, and the model is worthless.

In [ ]:
def spatial_features(res):
    """neighbour count + neighbour mean value from grid_disk(cell, 1)."""
    cl = np.array([h3.latlng_to_cell(a, o, res) for a, o in zip(lat, lon)])

    tot, cnt = defaultdict(float), defaultdict(int)
    for i in tr:                                 # TRAIN ROWS ONLY
        tot[cl[i]] += val[i]
        cnt[cl[i]] += 1

    nb_ct = np.zeros(n)
    nb_mn = np.full(n, np.nan)
    for i in range(n):
        s = 0.0; k = 0
        for nb in h3.grid_disk(cl[i], 1):        # the cell plus its six neighbours
            s += tot.get(nb, 0.0)
            k += cnt.get(nb, 0)
        if i in TRAIN:                           # a row must not see its own label
            s -= val[i]; k -= 1
        nb_ct[i] = k
        if k > 0:
            nb_mn[i] = s / k
    return nb_ct, nb_mn

nb_ct, nb_mn = spatial_features(RES)
X = df[BASE].copy()
X["nb_count"] = nb_ct
X["nb_mean_value"] = nb_mn

m2 = HistGradientBoostingRegressor(random_state=0).fit(X.iloc[tr], val[tr])
res7_rmse = rmse(val[te], m2.predict(X.iloc[te]))

print(f"baseline, no location   RMSE = ${BASE_RMSE:,.0f}")
print(f"with H3 neighbourhood   RMSE = ${res7_rmse:,.0f}")
print(f"                        improvement = {100*(BASE_RMSE-res7_rmse)/BASE_RMSE:.1f}%")

### Which resolution? — the question everyone asks second

Resolution 7 was not a lucky guess. The target is a property of the **block**, not of the cell,
so changing resolution changes only the *features* — which makes this a fair comparison.

In [ ]:
sweep = []
print(f"{'res':>4}{'edge km':>10}{'med nbrs':>10}{'coverage':>10}{'RMSE':>10}{'vs base':>9}")
for res in range(4, 11):
    ct, mn = spatial_features(res)
    Xs = df[BASE].copy(); Xs["nb_count"] = ct; Xs["nb_mean_value"] = mn
    mm = HistGradientBoostingRegressor(random_state=0).fit(Xs.iloc[tr], val[tr])
    r = rmse(val[te], mm.predict(Xs.iloc[te]))
    cov = 100 * float((ct > 0).mean())
    e = h3.average_hexagon_edge_length(res, unit="km")
    sweep.append((res, e, float(np.median(ct)), cov, r))
    print(f"{res:>4}{e:>10.3f}{np.median(ct):>10.0f}{cov:>9.1f}%{r:>10,.0f}"
          f"{100*(BASE_RMSE-r)/BASE_RMSE:>8.1f}%")

In [ ]:
res_ = [s[0] for s in sweep]
rmses = [s[4] for s in sweep]
covs = [s[3] for s in sweep]

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(res_, rmses, "o-", color="#c1440e", label="RMSE")
ax1.axhline(BASE_RMSE, ls="--", color="grey", label="no spatial feature")
ax1.set_xlabel("H3 resolution"); ax1.set_ylabel("RMSE ($)")
best = min(sweep, key=lambda s: s[4])
ax1.scatter([best[0]], [best[4]], s=180, facecolors="none", edgecolors="green", linewidths=2)

ax2 = ax1.twinx()
ax2.plot(res_, covs, "s--", color="#2b6cb0", alpha=.6, label="coverage %")
ax2.set_ylabel("% of rows with >=1 neighbour")

ax1.set_title(f"Too coarse, too fine, and the sweet spot (res {best[0]})")
fig.legend(loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.08))
ax1.grid(alpha=.3); plt.show()

A clean inverted U, and **both ends fail for opposite reasons**:

- **Resolution 4** — the cell is 26 km across, so a "neighbourhood" is most of the region. The
  feature is an average of everything, which is an average of nothing. It scores *worse than
  having no spatial feature at all.*
- **Resolutions 9 and 10** — the cell is smaller than the gap between blocks. Only **53%** of rows
  have even one neighbour, so the feature is missing for half the data. Note they score
  *identically*: the feature has died the same death twice.

### The rule of thumb worth writing down

> **Pick the finest resolution whose neighbourhood still holds enough members for a stable
> statistic — and watch COVERAGE, not cell size.**

At the optimum, coverage is still 93% and the median block has ~33 neighbours. When coverage
drops below about 90%, that is the warning light. This transfers to any dataset you own.

---
## Part 5 · Scoreboard

In [ ]:
print(f"{'':<28}{'traditional':>16}{'H3':>16}")
print("-" * 60)
print(f"{'Q1 exact neighbours':<28}{BRUTE_SECONDS:>15.1f}s{HYBRID_SECONDS:>15.1f}s")
print(f"{'  (same answer?)':<28}{'':>16}{'identical':>16}")
print(f"{'Q2 aggregate to areas':<28}{'round + groupby':>16}{'groupby':>16}")
print(f"{'cell shape':<28}{'changes with lat':>16}{'constant':>16}")
print(f"{'neighbours':<28}{'8, spread 1.60x':>16}{'6, spread 1.03x':>16}")
print(f"{'scaling':<28}{'O(n^2)':>16}{'~O(n)':>16}")
print(f"{'model RMSE':<28}{BASE_RMSE:>15,.0f}{res7_rmse:>16,.0f}")
print()
print("H3 didn't make the math faster.")
print("It made most of the math unnecessary.")

---
## Appendix A · Change the resolution and watch the map move

Every number in this notebook came from one variable. Set `RES = 9`, re-run Part 3, and the
hexagons shrink from 5 km² to 0.1 km².

In [ ]:
for r in (5, 7, 9, 11):
    cell = h3.latlng_to_cell(37.0, -120.0, r)
    print(f"res {r:>2}: {cell}  area {h3.cell_area(cell, unit='km^2'):9.4f} km2"
          f"   edge {h3.average_hexagon_edge_length(r, unit='km')*1000:8.1f} m")

## Appendix B · h3-py v3 → v4

Most tutorials online are still v3 and **will fail** on the version installed above.

| v3 | v4 (what we used) |
|---|---|
| `geo_to_h3` | `latlng_to_cell` |
| `h3_to_geo` | `cell_to_latlng` |
| `h3_to_geo_boundary` | `cell_to_boundary` |
| `k_ring` | `grid_disk` |
| `h3_distance` | `grid_distance` |
| `h3_to_parent` | `cell_to_parent` |
| `h3_to_children` | `cell_to_children` |
| `hex_area` | `average_hexagon_area` |
| `edge_length` | `average_hexagon_edge_length` |

## Appendix C · One caveat, said out loud

A random train/test split slightly flatters a neighbourhood feature, because neighbouring blocks
land on both sides of the split. Fine for a 20-minute lab; in production you would use a
**spatial** split — hold out whole regions, not random rows.